# RFM Customer Segmentation

UCI Online Retail II — about a million transaction rows across two years of a UK
online gift retailer. The aim is to turn raw transactions into a small number of
customer segments the marketing team can actually act on: who to protect, who to grow,
who to win back.

Approach: build Recency / Frequency / Monetary features per customer, then cluster with
K-Means. The interesting tension here is that the statistically "best" k isn't the most
useful k for the business — more on that below.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

!kaggle datasets download -d mashlyn/online-retail-ii-uci --unzip -q

# The download ships as a CSV despite the dataset title mentioning Excel
df = pd.read_csv('online_retail_II.csv', encoding='latin-1')
df.shape

In [ ]:
# Standard retail-data cleanup:
#  - invoices starting with 'C' are cancellations, not sales
#  - rows without a Customer ID can't be attributed to anyone
#  - negative/zero quantity or price are returns and data noise
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df = df[~df['Invoice'].astype(str).str.startswith('C')]
df = df.dropna(subset=['Customer ID'])
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]

df['Revenue'] = df['Quantity'] * df['Price']
df['Customer ID'] = df['Customer ID'].astype(int)

print(f"{len(df):,} clean transaction rows")
print(f"{df['Customer ID'].nunique():,} customers")
print(f"{df['InvoiceDate'].min().date()} to {df['InvoiceDate'].max().date()}")
print(f"${df['Revenue'].sum():,.0f} total revenue")

## Building RFM features

Recency is measured against the day after the last transaction in the data, so the most
recent buyers score close to zero days. Frequency counts distinct invoices, not line
items. Monetary is lifetime revenue.

In [ ]:
snapshot = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('Customer ID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot - x.max()).days),
    Frequency=('Invoice', 'nunique'),
    Monetary=('Revenue', 'sum'),
).reset_index()

rfm.describe().round(1)

The spread on Monetary is enormous — median around \$900 but a long tail into the
hundreds of thousands. That skew will dominate distance calculations, so log-transform
the features before scaling.

In [ ]:
rfm_log = rfm.copy()
for col in ['Recency', 'Frequency', 'Monetary']:
    rfm_log[col] = np.log1p(rfm_log[col])

X = StandardScaler().fit_transform(rfm_log[['Recency', 'Frequency', 'Monetary']])

## Choosing k

Check silhouette score across a range. In practice the top score here lands at k=2, which
splits the base into "active" and "lapsed" and nothing more — technically clean, useless
for targeting. k=4 gives a meaningfully lower-but-still-reasonable score and produces four
segments that map onto real marketing actions, so that's the call.

In [ ]:
for k in range(2, 9):
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X)
    print(f"k={k}: silhouette {silhouette_score(X, labels):.3f}")

In [ ]:
# Going with k=4 for business interpretability (see note above)
rfm['Cluster'] = KMeans(n_clusters=4, random_state=42, n_init=10).fit_predict(X)

profile = rfm.groupby('Cluster').agg(
    Customers=('Customer ID', 'size'),
    Recency=('Recency', 'mean'),
    Frequency=('Frequency', 'mean'),
    Monetary=('Monetary', 'mean'),
    Revenue=('Monetary', 'sum'),
).round(1)
profile

## Naming the segments

Read the cluster profiles and label them. The mapping is based on the means above:
recent + frequent + high-spend is Champions; recent but low-frequency is New/Promising;
long-since-seen with moderate history is At-Risk; dormant and low-value is Lost.

In [ ]:
# Map cluster id -> label by sorting on the profile, so the names follow the data
# rather than hardcoded cluster numbers (KMeans label order isn't stable across runs).
champ   = profile['Monetary'].idxmax()
lost    = profile['Recency'].idxmax()
remaining = [c for c in profile.index if c not in (champ, lost)]
# of the two middle clusters, the more recent one is the newer cohort
new_promising = min(remaining, key=lambda c: profile.loc[c, 'Recency'])
at_risk = [c for c in remaining if c != new_promising][0]

names = {champ: 'Champions', new_promising: 'New / Promising',
         at_risk: 'At-Risk', lost: 'Lost / Inactive'}
rfm['Segment'] = rfm['Cluster'].map(names)

summary = rfm.groupby('Segment').agg(
    Customers=('Customer ID', 'size'),
    Avg_Recency=('Recency', 'mean'),
    Avg_Frequency=('Frequency', 'mean'),
    Avg_Monetary=('Monetary', 'mean'),
    Revenue=('Monetary', 'sum'),
).round(1)
summary['Revenue_%'] = (summary['Revenue'] / summary['Revenue'].sum() * 100).round(1)
summary.sort_values('Revenue', ascending=False)

The headline falls straight out of this table: Champions are a fifth of customers but
roughly three-quarters of revenue. That concentration is the whole argument for a
retention-first budget.

## Charts

In [ ]:
BG = '#0D1117'
SEG_COLOR = {'Champions': '#00D4FF', 'New / Promising': '#00FF9D',
             'At-Risk': '#FFB830', 'Lost / Inactive': '#E84545'}
ORDER = ['Champions', 'New / Promising', 'At-Risk', 'Lost / Inactive']

def style_ax(ax):
    ax.set_facecolor(BG)
    for s in ['top', 'right']: ax.spines[s].set_visible(False)
    for s in ['bottom', 'left']: ax.spines[s].set_color('#30363D')
    ax.tick_params(colors='#8B949E')

counts = summary['Customers'].reindex(ORDER)
revenue = summary['Revenue'].reindex(ORDER)

In [ ]:
# Chart 1 — distribution and revenue share side by side
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.patch.set_facecolor(BG)
cols = [SEG_COLOR[s] for s in ORDER]

axes[0].pie(counts, labels=ORDER, colors=cols, autopct='%1.1f%%', startangle=90,
            wedgeprops=dict(width=0.55, edgecolor=BG, linewidth=3),
            textprops=dict(color='white', fontsize=9))
axes[0].set_title('Customers by segment', color='white', fontweight='bold', pad=12)

style_ax(axes[1])
axes[1].bar(ORDER, revenue / 1e6, color=cols, width=0.6)
axes[1].set_ylabel('Revenue ($M)', color='#8B949E')
axes[1].set_title('Revenue by segment', color='white', fontweight='bold', pad=12)
for i, v in enumerate(revenue / 1e6):
    pct = revenue.iloc[i] / revenue.sum() * 100
    axes[1].text(i, v + 0.1, f'${v:.1f}M\n{pct:.0f}%', ha='center',
                 color='white', fontweight='bold', fontsize=9)
for t in axes[1].get_xticklabels():
    t.set_rotation(15); t.set_fontsize(8)

ax = axes[2]; ax.axis('off'); ax.set_facecolor('#161B22')
champ_rev = summary.loc['Champions', 'Revenue'] / summary['Revenue'].sum() * 100
notes = [(f"{len(rfm):,}", 'customers', '#00D4FF'),
         (f"${rfm['Monetary'].sum()/1e6:.1f}M", 'revenue (2 yrs)', '#00FF9D'),
         (f'{champ_rev:.0f}%', 'revenue from Champions', '#00D4FF'),
         (f"{summary.loc['Lost / Inactive','Customers']/len(rfm)*100:.0f}%",
          'lost / inactive', '#E84545')]
for i, (num, lbl, c) in enumerate(notes):
    y = 0.82 - i * 0.22
    ax.text(0.5, y, num, ha='center', fontsize=22, fontweight='bold', color=c)
    ax.text(0.5, y - 0.08, lbl, ha='center', fontsize=9, color='#8B949E')
ax.set_title('Headline numbers', color='white', fontweight='bold', pad=12)

plt.tight_layout()
plt.savefig('chart_01_segment_overview.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

In [ ]:
# Chart 2 — segments in 3D RFM space (sampled for legibility)
from mpl_toolkits.mplot3d import Axes3D
fig = plt.figure(figsize=(13, 9)); fig.patch.set_facecolor(BG)
ax = fig.add_subplot(111, projection='3d'); ax.set_facecolor(BG)

for seg in ORDER:
    sub = rfm[rfm['Segment'] == seg]
    sub = sub.sample(min(300, len(sub)), random_state=42)
    ax.scatter(sub['Recency'], sub['Frequency'], np.log1p(sub['Monetary']),
               c=SEG_COLOR[seg], label=f'{seg} (n={(rfm.Segment==seg).sum():,})',
               alpha=0.6, s=18)
ax.set_xlabel('Recency (days)', color='#8B949E')
ax.set_ylabel('Frequency', color='#8B949E')
ax.set_zlabel('log Monetary', color='#8B949E')
ax.tick_params(colors='#8B949E', labelsize=7)
ax.legend(facecolor='#161B22', labelcolor='white', edgecolor='#30363D', loc='upper left')
ax.set_title('Segments in RFM space', color='white', fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('chart_02_3d_scatter.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

In [ ]:
# Chart 3 — normalized RFM profile per segment, as small-multiple radars
labels_axis = ['Recency\n(fresh)', 'Frequency', 'Monetary']
norm = summary[['Avg_Recency', 'Avg_Frequency', 'Avg_Monetary']].copy()
# invert recency so "more is better" holds on all three axes
norm['Avg_Recency'] = norm['Avg_Recency'].max() - norm['Avg_Recency']
norm = (norm - norm.min()) / (norm.max() - norm.min())

angles = np.linspace(0, 2*np.pi, 3, endpoint=False).tolist()
angles += angles[:1]
fig, axes = plt.subplots(1, 4, figsize=(20, 6), subplot_kw=dict(polar=True))
fig.patch.set_facecolor(BG)
for ax, seg in zip(axes, ORDER):
    ax.set_facecolor(BG)
    vals = norm.loc[seg].tolist(); vals += vals[:1]
    ax.plot(angles, vals, color=SEG_COLOR[seg], linewidth=2)
    ax.fill(angles, vals, color=SEG_COLOR[seg], alpha=0.25)
    ax.set_xticks(angles[:-1]); ax.set_xticklabels(labels_axis, color='white', size=8)
    ax.set_yticklabels([]); ax.grid(color='#30363D')
    rev = summary.loc[seg, 'Revenue'] / 1e6
    ax.set_title(f"{seg}\n{summary.loc[seg,'Customers']:,.0f} cust · ${rev:.1f}M",
                 color=SEG_COLOR[seg], fontweight='bold', size=10, pad=14)
plt.tight_layout()
plt.savefig('chart_03_radar_profiles.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

In [ ]:
# Chart 4 — recency vs value, colored by segment, with the action framing
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8)); fig.patch.set_facecolor(BG)
style_ax(ax1)
for seg in ORDER:
    sub = rfm[rfm['Segment'] == seg].sample(
        min(400, (rfm.Segment == seg).sum()), random_state=42)
    ax1.scatter(sub['Recency'], np.log1p(sub['Monetary']),
                c=SEG_COLOR[seg], label=seg, alpha=0.5, s=22)
ax1.set_xlabel('Recency (days since last order)', color='#8B949E')
ax1.set_ylabel('log Monetary', color='#8B949E')
ax1.set_title('Recency vs value', color='white', fontweight='bold', pad=12)
ax1.legend(facecolor='#161B22', labelcolor='white', edgecolor='#30363D')

ax2.axis('off')
actions = {
    'Champions': 'Protect. Loyalty perks, early access. Highest cost to replace.',
    'New / Promising': 'Grow. Onboard and drive the second and third purchase.',
    'At-Risk': 'Win back. Targeted reactivation while the relationship is warm.',
    'Lost / Inactive': 'Triage. One last-chance offer, else stop spending here.',
}
y = 0.9
for seg in ORDER:
    ax2.text(0.02, y, seg, color=SEG_COLOR[seg], fontweight='bold', fontsize=12,
             transform=ax2.transAxes)
    row = summary.loc[seg]
    ax2.text(0.02, y - 0.05,
             f"{row['Customers']:,.0f} customers · ${row['Revenue']/1e6:.1f}M · "
             f"last seen {row['Avg_Recency']:.0f}d",
             color='#8B949E', fontsize=9, transform=ax2.transAxes)
    ax2.text(0.02, y - 0.095, actions[seg], color='white', fontsize=9,
             style='italic', transform=ax2.transAxes)
    y -= 0.24
ax2.set_title('What to do with each segment', color='white', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('chart_04_action_matrix.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

## Takeaways

- Champions (~20% of customers) drive ~74% of revenue — retention beats acquisition here
  on pure ROI.
- New / Promising buy as recently as Champions but far less often; the onboarding window
  is the growth lever.
- At-Risk still hold real historic value and haven't been gone long — the highest-return
  reactivation target.
- The k=2-vs-k=4 decision is the analytical judgment call: the useful answer wasn't the
  one with the best silhouette score.